# 2. ETL Pipeline

In [0]:
# Databricks widget parameter for dynamic city execution
dbutils.widgets.text("city", "toronto")
city_param = dbutils.widgets.get("city").strip().lower()

assert city_param in ["toronto", "vancouver", "montreal"], f"Invalid city parameter: {city_param}"
print(f"=== Starting Medallion ETL Pipeline for City: '{city_param.upper()}' ===")

# Create Lakehouse database metastore if not exists
spark.sql("CREATE DATABASE IF NOT EXISTS airbnb_lakehouse")

## 2.1 Bronze Layer Creation

Raw Ingestion

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Dynamic raw file path construction
raw_file_path = f"/Workspace/Shared/canada-airbnb/data/{city_param}/listings.csv.gz"

print(f"[BRONZE] Reading raw listings data from: {raw_file_path}")

# Load raw compressed CSV
df_bronze = spark.read.option("header", "true") \
                       .option("multiLine", "true") \
                       .option("escape", '"') \
                       .csv(raw_file_path)

# Tag record with target city name and ingestion timestamp
df_bronze = df_bronze.withColumn("city", F.lit(city_param)) \
                     .withColumn("ingested_at", F.current_timestamp())

# Register/Save Bronze Table
bronze_table_name = f"airbnb_lakehouse.bronze_listings_{city_param}"
df_bronze.write.format("delta").mode("overwrite").saveAsTable(bronze_table_name)

print(f"[BRONZE] Successfully loaded {df_bronze.count()} records into table: {bronze_table_name}")

## 2.2 Silver Layer

Cleaning, Type Casting & Imputation

In [0]:
print(f"[SILVER] Processing data cleaning and feature transformations...")

# Helper function to safely handle empty strings ('') and invalid formats during type casting
def safe_cast(col_or_name, target_type):
    c = col_or_name if isinstance(col_or_name, F.Column) else F.col(col_or_name)
    # Replace empty strings or whitespace with NULL before casting to prevent ANSI cast errors
    c_clean = F.when(F.trim(c) == "", None).otherwise(c)
    return c_clean.cast(target_type)

# 1. Clean Price column ($150.00 -> 150.0) safely
raw_price_cleaned = F.regexp_replace(F.col("price"), r"[\$,]", "")
df_silver = df_bronze.withColumn("price_clean", safe_cast(raw_price_cleaned, "double"))

# 2. Extract Bathrooms numeric value from bathrooms_text if bathrooms is null or empty string
extracted_bathrooms = F.regexp_extract(F.col("bathrooms_text"), r"(\d+\.?\d*)", 1)
df_silver = df_silver.withColumn(
    "bathrooms_clean",
    F.coalesce(
        safe_cast("bathrooms", "double"),
        safe_cast(extracted_bathrooms, "double"),
        F.lit(1.0)
    )
)

# 3. Clean numeric columns & cast types safely
df_silver = df_silver \
    .withColumn("accommodates", safe_cast("accommodates", "int")) \
    .withColumn("bedrooms", F.coalesce(safe_cast("bedrooms", "int"), F.lit(1))) \
    .withColumn("beds", F.coalesce(safe_cast("beds", "int"), F.lit(1))) \
    .withColumn("latitude", safe_cast("latitude", "double")) \
    .withColumn("longitude", safe_cast("longitude", "double")) \
    .withColumn("availability_365", F.coalesce(safe_cast("availability_365", "int"), F.lit(0))) \
    .withColumn("minimum_nights", F.coalesce(safe_cast("minimum_nights", "int"), F.lit(1))) \
    .withColumn("maximum_nights", F.coalesce(safe_cast("maximum_nights", "int"), F.lit(365))) \
    .withColumn("hosts_time_as_host_years", F.coalesce(safe_cast("hosts_time_as_host_years", "int"), F.lit(0))) \
    .withColumn("hosts_time_as_host_months", F.coalesce(safe_cast("hosts_time_as_host_months", "int"), F.lit(0)))

# 4. Standardize Categorical Columns
df_silver = df_silver \
    .withColumn("room_type", F.lower(F.trim(F.coalesce(F.col("room_type"), F.lit("entire home/apt"))))) \
    .withColumn("property_type", F.lower(F.trim(F.coalesce(F.col("property_type"), F.lit("other"))))) \
    .withColumn("neighbourhood_cleansed", F.coalesce(F.col("neighbourhood_cleansed"), F.lit("Unknown"))) \
    .withColumn("host_is_superhost", F.coalesce(F.col("host_is_superhost"), F.lit("f")))

# 5. Outlier & Invalid Data Filtering (Price > 0 and Price <= 2500)
df_silver = df_silver.filter(
    (F.col("price_clean").isNotNull()) & 
    (F.col("price_clean") > 0) & 
    (F.col("price_clean") <= 2500)
)

# 6. Exclude 100% missing and prohibited leaked columns
leaked_and_empty_cols = [
    "neighborhood_overview", "host_since", "host_response_time", "host_response_rate",
    "host_acceptance_rate", "host_thumbnail_url", "host_neighbourhood", "host_total_listings_count",
    "weekly_price", "monthly_price", "license", "calendar_updated", "picture_url", "host_url"
]

silver_cols = [c for c in df_silver.columns if c not in leaked_and_empty_cols]
df_silver_clean = df_silver.select(silver_cols)

# Save to Partitioned Silver Delta Table
silver_table_name = "airbnb_lakehouse.silver_listings"

df_silver_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", f"city = '{city_param}'") \
    .saveAsTable(silver_table_name)

print(f"[SILVER] Saved {df_silver_clean.count()} cleaned records to '{silver_table_name}' for city '{city_param}'")

## 2.3 Gold Layer

ML Feature Table & Analytics Summary

In [0]:
print(f"[GOLD] Generating Gold feature tables and BI metric summaries...")

# --- 1. GOLD FEATURE TABLE FOR ML MODEL (Person 2 Input) ---
gold_ml_features = df_silver_clean.select(
    "id",
    "city",
    "price_clean",
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_clean",
    "latitude",
    "longitude",
    "neighbourhood_cleansed",
    "hosts_time_as_host_years",
    "hosts_time_as_host_months",
    "host_is_superhost",
    "minimum_nights",
    "availability_365"
)

gold_ml_table = "airbnb_lakehouse.gold_listings_features"
gold_ml_features.write \
    .format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", f"city = '{city_param}'") \
    .saveAsTable(gold_ml_table)

print(f"[GOLD] Saved ML Features Table to '{gold_ml_table}' ({gold_ml_features.count()} rows)")

# --- 2. GOLD METRICS TABLE FOR DASHBOARD & GENIE AGENT (Person 3 Input) ---
gold_city_summary = df_silver_clean.groupBy("city", "room_type").agg(
    F.count("id").alias("total_listings"),
    F.round(F.expr("percentile_approx(price_clean, 0.5)"), 2).alias("median_price"),
    F.round(F.mean("price_clean"), 2).alias("avg_price"),
    F.round(F.mean("availability_365"), 1).alias("avg_availability"),
    F.round(F.mean("accommodates"), 1).alias("avg_accommodates")
)

gold_summary_table = "airbnb_lakehouse.gold_city_metrics"
gold_city_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", f"city = '{city_param}'") \
    .saveAsTable(gold_summary_table)

print(f"[GOLD] Saved Aggregated Metrics to '{gold_summary_table}'")
print(f"=== ETL Pipeline Execution Completed Successfully for {city_param.upper()} ===")

### Check the correctness of layers creation

In [0]:
%sql
SELECT city, count(*) as count, min(price_clean) as min_p, max(price_clean) as max_p 
FROM airbnb_lakehouse.silver_listings 
GROUP BY city;

## Architecture Overview:

Our Lakehouse architecture follows the standard Medallion Pattern built on Top of Delta Lake to ensure structured data processing, analytics, and machine learning scalability:
- Bronze Layer (Raw Ingestion): Stores unmodified, raw listings CSV data read directly from compressed files per city. It acts as an immutable historical append log with added metadata (city, ingested_at).
- Silver Layer (Cleaned & Typed): Standardizes field data types (converts string prices to numeric floats), imputes missing property attributes, filters extreme price outliers ($> \$2,500$), and enforces Data Leakage rules by dropping post-booking or price-derived attributes.- Gold Layer (Business & ML Support): Contains highly refined output views tailored for downstream tasks:

1. gold_listings_features: Optimized input dataset for ML training.
2. gold_city_metrics: Aggregated statistical metrics supporting BI dashboards and natural language querying via Genie Agent.
3. gold_model_predictions: Gold-standard table mapping baseline predictions, actual prices, and model error metrics.